# II : Cleaned Data Validation

This notebook validates the calibration-candidate dataset exported by `01_Market_Data_Cleaning.ipynb`. It does not repeat the raw-data cleaning steps.

## 1. Project paths and imports

Load the libraries and locate the compressed calibration dataset using a project-relative path.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "cleaned" / "calibration_candidates.csv.gz"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Calibration dataset not found: {DATA_PATH}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")

## 2. Load the exported dataset

Read the long-format CSV. Each row should represent one option contract from the selected positive-volume sample.

In [ ]:
options = pd.read_csv(DATA_PATH)
print(f"Rows: {len(options):,} | Columns: {len(options.columns)}")
display(options.head())

## 3. Validate the canonical schema

Check that the fields required by the calibration workflow are present. The expiry-day field is also retained for maturity-coverage diagnostics.

In [ ]:
REQUIRED_COLUMNS = {
    "Expiry",
    "Days_To_Maturity",
    "Strike",
    "Tau",
    "Spot",
    "Dividend_Yield",
    "Risk_Free_Rate_CC",
    "Market_Price",
    "Trading_Volume",
    "Implied_Volatility",
    "Relative_Spread",
}
missing_columns = REQUIRED_COLUMNS.difference(options.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print("Schema check passed.")
display(options[list(sorted(REQUIRED_COLUMNS))].dtypes)

## 4. Check sample size and maturity coverage

Summarise the number of observations by expiry and maturity. The expected primary sample contains 477 observations across five expiries; the code reports the actual file contents.

In [ ]:
maturity_summary = (
    options.groupby(["Expiry", "Days_To_Maturity"], dropna=False)
    .size()
    .rename("Number_of_Quotes")
    .reset_index()
    .sort_values("Days_To_Maturity")
)
print(f"Total observations: {len(options):,}")
print(f"Distinct expiries: {options['Expiry'].nunique()}")
display(maturity_summary)

## 5. Numerical and economic consistency checks

Check that the main numerical inputs are finite and economically admissible, and verify that the sample contains only contracts with positive trading volume.

In [ ]:
numeric_columns = [
    "Days_To_Maturity", "Strike", "Tau", "Spot", "Dividend_Yield",
    "Risk_Free_Rate_CC", "Market_Price", "Trading_Volume",
    "Implied_Volatility", "Relative_Spread",
]
non_finite_counts = options[numeric_columns].apply(
    lambda column: (~np.isfinite(column)).sum()
)
checks = {
    "all_strikes_positive": bool((options["Strike"] > 0).all()),
    "all_days_to_maturity_positive": bool((options["Days_To_Maturity"] > 0).all()),
    "all_tau_positive": bool((options["Tau"] > 0).all()),
    "all_spot_values_positive": bool((options["Spot"] > 0).all()),
    "all_market_prices_non_negative": bool((options["Market_Price"] >= 0).all()),
    "all_trading_volumes_positive": bool((options["Trading_Volume"] > 0).all()),
    "all_numeric_inputs_finite": bool((non_finite_counts == 0).all()),
}
display(non_finite_counts.rename("Non_Finite_Count").to_frame())
display(pd.Series(checks, name="Passed"))
if not all(checks.values()):
    raise ValueError("One or more calibration input checks failed. Inspect the diagnostics above.")
print("All input validation checks passed.")

## 6. Calibration sample snapshot

Record the key dimensions and ranges of the dataset immediately before calibration.

In [ ]:
sample_snapshot = pd.Series({
    "Number of calibration quotes": len(options),
    "Number of expiries": options["Expiry"].nunique(),
    "Minimum maturity (days)": options["Days_To_Maturity"].min(),
    "Maximum maturity (days)": options["Days_To_Maturity"].max(),
    "Minimum strike": options["Strike"].min(),
    "Maximum strike": options["Strike"].max(),
    "Minimum market price": options["Market_Price"].min(),
    "Maximum market price": options["Market_Price"].max(),
})
display(sample_snapshot.to_frame("Value"))